# 3.2. Validation: Pathway signatures
# 3.2. 模型验证：通路特征

### 📘 Overview
### 📘 概述

This notebook uses DESeq2 differential-expression results to perform pathway enrichment analysis with WikiPathways gene sets, producing –log₁₀ *p*-values that quantify pathway activation for each compound.
这个notebook使用验证数据的DESeq2差异表达结果，通过WikiPathways基因集进行通路富集分析，生成–log₁₀ p值来量化每个化合物的通路激活程度。

**Inputs**  
**输入数据**  
DESeq2 differential expression data
DESeq2差异表达数据（验证数据）

**Output**  
**输出**  
An AnnData file with –log₁₀ *p*-values for each pathway–compound pair
包含每个通路-化合物对的–log₁₀ p值的AnnData文件

In [ ]:
%%capture

!conda install -c bioconda gseapy

In [ ]:
import numpy as np


import dilimap as dmap

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
dmap.logging.print_version()

Running dilimap 0.2.dev20+gc285bd842 (python 3.10.19) on 2025-11-20 02:20.


## Run pathway enrichment analysis
## 运行通路富集分析

使用WikiPathways基因集对验证数据的DESeq2结果进行通路富集分析，计算每个通路的激活分数。

In [ ]:
adata_deseq = dmap.s3.read('validation_data_deseq2.h5ad')
# 从S3读取验证数据的DESeq2结果

Package: s3://dilimap/public/data. Top hash: 155a2b3b63


In [ ]:
FDR = adata_deseq.to_df('FDR')
# 从DESeq2结果中提取调整后的p值（FDR = False Discovery Rate，错误发现率）

Creating directory /Users/callustang/Library/Application Support/bioservices 


[autoreload of gseapy.gsea failed: Traceback (most recent call last):
  File "/opt/miniconda3/envs/dilimap/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/opt/miniconda3/envs/dilimap/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/opt/miniconda3/envs/dilimap/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/opt/miniconda3/envs/dilimap/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
  File "/opt/miniconda3/envs/dilimap/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/opt/miniconda3/envs/dilimap/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 309, in update_function
    setattr(

In [ ]:
# 尝试计算通路特征
# 注意：如果遇到 gseapy 版本兼容性问题，可以尝试从 S3 加载预计算的结果
# 计算通路特征：使用WikiPathways基因集进行富集分析

try:
    adata_wiki = dmap.pp.pathway_signatures(FDR)
    # 计算通路特征：输入调整后的p值矩阵（基因 × 化合物），输出通路激活分数矩阵（通路 × 化合物），值为–log₁₀ p值
    print("✓ 成功计算通路特征")
except Exception as e:
    error_msg = str(e)
    print("=" * 80)
    print("❌ 计算通路特征时出错")
    print("=" * 80)
    print(f"\n错误信息：{error_msg}\n")
    
    # 检查是否是 gseapy 版本问题
    if "organism" in error_msg.lower() or "enrichr" in error_msg.lower():
        print("可能的原因：gseapy 版本不兼容")
        print("解决方案：")
        print("  1. 尝试更新 gseapy:")
        print("     !pip install --upgrade gseapy")
        print("  2. 或尝试从 S3 加载预计算的结果（如果可用）\n")
    
    # 尝试从 S3 加载预计算的结果
    print("尝试从 S3 加载预计算的验证数据通路特征...")
    try:
        adata_wiki = dmap.s3.read('validation_data_pathways.h5ad')
        # 如果计算失败，尝试从S3加载预计算的结果
        print("✓ 成功从 S3 加载预计算的通路特征数据")
    except Exception as s3_e:
        print(f"✗ 无法从 S3 加载: {s3_e}\n")
        print("=" * 80)
        raise RuntimeError(
            "无法计算或加载通路特征数据。\n"
            "请检查：\n"
            "1. gseapy 版本是否正确安装\n"
            "2. 网络连接是否正常\n"
            "3. 是否有预计算的数据可用"
        ) from e

Attempt 1 failed with error: 'DataFrame' object has no attribute 'append'
Attempt 2 failed with error: 'DataFrame' object has no attribute 'append'
Attempt 3 failed with error: 'DataFrame' object has no attribute 'append'
Attempt 4 failed with error: 'DataFrame' object has no attribute 'append'
Attempt 5 failed with error: 'DataFrame' object has no attribute 'append'
Failed after 5 attempts for index 0.
Attempt 1 failed with error: 'DataFrame' object has no attribute 'append'
Attempt 2 failed with error: 'DataFrame' object has no attribute 'append'
Attempt 3 failed with error: 'DataFrame' object has no attribute 'append'
Attempt 4 failed with error: 'DataFrame' object has no attribute 'append'
Attempt 5 failed with error: 'DataFrame' object has no attribute 'append'
Failed after 5 attempts for index 1.
Attempt 1 failed with error: 'DataFrame' object has no attribute 'append'
Attempt 2 failed with error: 'DataFrame' object has no attribute 'append'
Attempt 3 failed with error: 'DataFram

In [ ]:
# 检查 adata_wiki 是否已定义
if 'adata_wiki' not in globals():
    raise NameError(
        "错误：'adata_wiki' 未定义。\n"
        "请确保 Cell 8 已成功运行。\n"
        "如果遇到错误，请检查错误信息并按照提示解决。"
    )

# 复制观察数据
adata_wiki.obs = adata_deseq.obs.copy()

NameError: name 'adata_wiki' is not defined

## Mapping metadata and clinical annotations

In [ ]:
## Simplify keys

# 检查 adata_wiki 是否已定义
if 'adata_wiki' not in globals():
    raise NameError(
        "错误：'adata_wiki' 未定义。\n"
        "请确保前面的 cells 已成功运行，特别是 Cell 8（通路特征计算）。"
    )

adata_wiki.obs.rename(
    columns={
        'CONCENTRATION_UM': 'dose_uM',
        'DOSE_LEVEL': 'dose_level',
        'COMPOUND': 'compound_name',
    },
    inplace=True,
)

adata_wiki.obs['compound_name'] = np.where(
    adata_wiki.obs['compound_name'].isna(),
    adata_wiki.obs_names,
    adata_wiki.obs['compound_name'],
)

adata_wiki.obs_names = adata_wiki.obs_names.str.replace('CPZ', 'Chlorpromazine')
adata_wiki.obs.loc[adata_wiki.obs['compound_name'] == 'Ibrutinib', 'SPLIT'] = 'training'

NameError: name 'adata_wiki' is not defined

In [ ]:
## Cmax annotations
obs_names = adata_wiki.obs_names.str.lower()

df_CMAX = dmap.s3.read('compound_Cmax_values.csv')
df_CMAX.index = df_CMAX.index.str.lower()
adata_wiki.obs['Cmax_uM'] = obs_names.map(df_CMAX['Cmax_median'])

## DILI annotations
df_DILI = dmap.s3.read('compound_DILI_labels.csv')
df_DILI.index = df_DILI.index.str.lower()
for col in df_DILI.columns:
    adata_wiki.obs[col] = obs_names.map(df_DILI[col])

adata_wiki.obs['DILIrank'] = adata_wiki.obs['DILIrank'].replace(np.nan, '')
adata_wiki.obs['livertox_score'] = adata_wiki.obs['livertox_score'].replace(np.nan, '')

## Viability (LDH) IC10 annotations
df_LDH = dmap.s3.read('compound_cell_viability_IC10.csv')
df_LDH.index = df_LDH.index.str.lower()
for k in df_LDH.columns:
    adata_wiki.obs[f'LDH_{k}'] = obs_names.map(df_LDH[k])

## Number DEGs
adata_wiki.obs['n_DEG'] = adata_deseq.obs['n_DEG'] = (
    adata_deseq.layers['FDR'] < 0.05
).sum(1)

NameError: name 'adata_wiki' is not defined

In [ ]:
set(
    adata_wiki.obs_names[
        adata_wiki.obs['Cmax_uM'].isna() | adata_wiki.obs['DILI_label'].isna()
    ]
)

NameError: name 'adata_wiki' is not defined

In [ ]:
adata_wiki = adata_wiki[
    ~(adata_wiki.obs['Cmax_uM'].isna() | adata_wiki.obs['DILI_label'].isna())
].copy()

NameError: name 'adata_wiki' is not defined

In [ ]:
adata_wiki.X = np.nan_to_num(adata_wiki.X)  # set nans to zero

NameError: name 'adata_wiki' is not defined

## Push file to S3

In [ ]:
# dmap.s3.write(adata_wiki, 'validation_data_pathways.h5ad', package_name='public/data')